Para la notebook de "Inferencia Ecológica para Evolución del Voto", podemos estructurar el trabajo en varias secciones para realizar un análisis completo y sistemático. Aquí te propongo una estructura posible para la notebook:

### 1. Introducción y Objetivos
   - Breve descripción de lo que es la inferencia ecológica y por qué es relevante para el análisis de datos electorales.
   - Definición de los objetivos específicos de la notebook.

### 2. Preparación de Datos
   - **Carga de Datos**: Importar los datos electorales necesarios para el análisis.
   - **Limpieza de Datos**: Verificar la calidad de los datos y realizar cualquier limpieza o transformación necesaria.
   - **Verificación de Datos**: Asegurarse de que los datos están correctamente estructurados y listos para el análisis.

### 3. Análisis Exploratorio de Datos (EDA)
   - **Resumen Estadístico**: Proporcionar estadísticas descriptivas de los datos.
   - **Visualización de Datos**: Crear gráficos para visualizar las tendencias y patrones en los datos.
   - **Identificación de Tendencias**: Observar cómo los votos para los diferentes partidos o candidatos han evolucionado a lo largo del tiempo.

### 4. Modelado y Inferencia
   - **Selección de Modelo**: Elegir el modelo o los modelos estadísticos apropiados para la inferencia ecológica.
   - **Ajuste de Modelo**: Ajustar el modelo a los datos.
   - **Evaluación de Modelo**: Evaluar la precisión y validez del modelo.
   - **Inferencias**: Realizar inferencias sobre la evolución del voto y los factores que pueden haber contribuido a los cambios observados.

### 5. Interpretación de Resultados
   - **Análisis de Resultados**: Interpretar los resultados del modelo y las inferencias realizadas.
   - **Identificación de Factores Clave**: Destacar los factores que parecen haber tenido un impacto significativo en la evolución del voto.

### 6. Visualización de Resultados
   - Crear visualizaciones para presentar los resultados de una manera clara y comprensible.

### 7. Conclusiones y Recomendaciones
   - Resumir los hallazgos clave.
   - Proporcionar recomendaciones basadas en los resultados.

### 8. Referencias
   - Citar cualquier fuente o recurso utilizado en la notebook.

### Herramientas y Bibliotecas Sugeridas
   - Pandas para la manipulación y análisis de datos.
   - Matplotlib y Seaborn para la visualización de datos.
   - Scikit-learn o Statsmodels para el modelado estadístico.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
path_datosBD = '/content/drive/My Drive/electoral/datos/BD'
# path_datosBD = './../datos/BD'



### 2. Preparación de Datos
   - **Carga de Datos**: Importar los datos electorales necesarios para el análisis.

In [ ]:
import pandas as pd

In [ ]:
pd.read_csv(path_datosBD+'/seccion_table.csv')

,distrito_id,seccion_id,seccionprovincial_id,seccion_nombre
0,1,15,0.0,Comuna 15
1,1,1,0.0,Comuna 01
2,1,4,0.0,Comuna 04
3,4,1,0.0,Capital
4,4,26,0.0,Unión
...,...,...,...,...
527,10,13,0.0,Rinconada
528,5,25,0.0,Berón De Astrada
529,11,7,0.0,Chical-Co
530,17,6,0.0,Rivadavia


In [ ]:

df_mesas = pd.read_csv(path_datosBD+'/mesas_table.csv')


### 2. Preparación de Datos

#### Carga de Datos:
- Se cargan los datos de votación de diferentes elecciones y otros archivos relacionados con la elección, como listas de agrupaciones y claves de departamentos.
- Se concatenan los DataFrames de diferentes elecciones y se filtran según los cargos electivos.



In [ ]:
import pandas as pd
# # ----- Data Loading -----

# Define the path to your data
# path_datosBD = "./your/data/path"
path_datosBD = '/content/drive/My Drive/electoral/datos/BD'

# Define the cargo IDs you are interested in
cargo_ids = [1, 3, 4]

# List of election files
election_files = ['votos_eleccion_17_table.csv', 'votos_eleccion_18_table.csv'] # df23PASO, df23GRAL

# Load and filter the data in a single line
df = pd.concat([pd.read_csv(f"{path_datosBD}/{file_name}").query("cargo_id in @cargo_ids") for file_name in election_files], axis=0)
df = df.drop(['agrupacion_nombre'], axis = 1)


<ipython-input-6-41360167c59e>:15: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f"{path_datosBD}/{file_name}").query("cargo_id in @cargo_ids") for file_name in election_files], axis=0)
<ipython-input-6-41360167c59e>:15: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f"{path_datosBD}/{file_name}").query("cargo_id in @cargo_ids") for file_name in election_files], axis=0)


In [ ]:


# df23PASO = pd.read_csv(path_datosBD + '/votos_eleccion_17_table.csv')
# df23GRAL = pd.read_csv(path_datosBD + '/votos_eleccion_18_table.csv')
# df = pd.concat([df23PASO, df23GRAL], axis=0)
# df = df.loc[(df.cargo_id.isin([1, 3, 4]))]

cargo = pd.read_csv(path_datosBD + '/cargo_tags.csv')
agrup_lista = pd.read_csv(path_datosBD + '/agrupacion_lista_table.csv').drop(['agrupacion_nombre'], axis = 1)
agrup_nombre = pd.read_csv('/content/drive/My Drive/electoral/datos/BD2/agrupacion_nombre_table.csv', dtype={'agrupacion_id': int}) ## agrupacion nombres
claves_dptos = pd.read_csv(path_datosBD + '/claves_dptos_ref.csv')
eleccion_tags = pd.read_csv(path_datosBD + '/eleccion_tags.csv')
prov_nams = pd.read_csv(path_datosBD + '/distrito_table.csv')



#### Limpieza de Datos:
- Se realizan transformaciones en los IDs de agrupación y números de lista para armonizarlos y asegurar consistencia.
- Se limpian y armonizan los nombres de las agrupaciones.


In [ ]:
import pandas as pd

# ----- Function Definitions -----
def harmonize_agrupacion_id(agrupacion_id):
    if pd.isna(agrupacion_id):
        return "000000"
    else:
        try:
            return str(int(float(agrupacion_id))).zfill(6)
        except ValueError:
            return agrupacion_id

def harmonize_agrupacion_names(df, col_name='agrupacion_nombre'):
    """Harmonizes the names of agrupaciones."""
    replacements = {
        'CAMBIEMOS BUENOS AIRES': 'CAMBIEMOS',
        'JUNTOS': 'JUNTOS POR EL CAMBIO'
    }
    df[col_name] = df[col_name].replace(replacements).str.title().str.strip()
    return df


In [ ]:

# Agrup Nombre Data Preprocessing
# agrup_nombre['agrupacion_id'] = agrup_nombre['agrupacion_id'].apply(harmonize_agrupacion_id)
merged_data = df.merge(agrup_nombre.drop(['eleccion_id'], axis = 1).drop_duplicates())
merged_data = harmonize_agrupacion_names(merged_data)


In [ ]:

# # # Agrup Lista Data Preprocessing
# # agrup_lista['agrupacion_id'] = agrup_lista['agrupacion_id'].apply(harmonize_agrupacion_id)
# # agrup_lista['lista_numero'] = agrup_lista['lista_numero'].apply(harmonize_agrupacion_id)


# # ----- Data Merging & Transformations -----
# # Merge with simil_nombre
# simil_nombre = agrup_lista.groupby(['eleccion_id', 'distrito_id', 'agrupacion_id']).agrupacion_nombre.first().reset_index()
# df['agrupacion_id'] = df['agrupacion_id'].astype(str).str.zfill(6)
# merged_data = df.merge(simil_nombre)
# merged_data = harmonize_agrupacion_names(merged_data)
# # display(merged_data)


In [ ]:
merged_data.head()

,distrito_id,seccionprovincial_id,seccion_id,circuito_id,mesa_id,cargo_id,agrupacion_id,lista_numero,votos_tipo,votos_cantidad,eleccion_id,agrupacion_nombre
0,1,0.0,1,000001,1,3,554,3111.0,POSITIVO,71,17,Alianza Union Por La Patria
1,1,0.0,1,000001,2,3,554,3111.0,POSITIVO,70,17,Alianza Union Por La Patria
2,1,0.0,1,000001,3,3,554,3111.0,POSITIVO,61,17,Alianza Union Por La Patria
3,1,0.0,1,000001,4,3,554,3111.0,POSITIVO,90,17,Alianza Union Por La Patria
4,1,0.0,1,000001,5,3,554,3111.0,POSITIVO,73,17,Alianza Union Por La Patria



#### Verificación de Datos:
- Aunque no se muestra explícitamente en el código proporcionado, se asume que las operaciones de fusión, agrupación y transformación de datos forman parte del proceso de verificación de datos para asegurarse de que están estructurados correctamente y listos para el análisis.

**Agrupa en Nivel Circuitos**

In [ ]:

# Group and aggregate data
# aggregated_data = merged_data.groupby(['eleccion_id', 'cargo_id', 'agrupacion_nombre', 'lista_numero', 'votos_tipo']).agg({'votos_cantidad': 'sum'}).reset_index()
aggregated_data = merged_data.groupby(['eleccion_id', 'cargo_id', 'agrupacion_nombre', 'votos_tipo']).agg({'votos_cantidad': 'sum'}).reset_index()
# display(aggregated_data)

# Top N aggregation
N = 20
top_aggregated_data = aggregated_data.groupby(['eleccion_id', 'cargo_id', 'votos_tipo'])\
                                     .apply(lambda x: x.nlargest(N, 'votos_cantidad'))\
                                     .reset_index(drop=True).rename(columns={'votos_cantidad': 'votos_nacional'})
# display(top_aggregated_data)


In [ ]:

# Further transformations on original data
data_copy = merged_data.copy()
data_copy = data_copy.merge(top_aggregated_data, how='left')
data_copy['agrupacion_nombre_'] = data_copy['agrupacion_nombre'].mask(data_copy['votos_nacional'].isnull(), 'Resto')


In [ ]:

# data_aggregated = data_copy.groupby(['distrito_id', 'seccion_id', 'circuito_id', 'mesa_id', 'cargo_id', 'agrupacion_nombre_', 'lista_numero', 'votos_tipo', 'eleccion_id'])\
data_aggregated = data_copy.groupby(['distrito_id', 'seccion_id', 'circuito_id', 'mesa_id', 'cargo_id', 'agrupacion_nombre_', 'votos_tipo', 'eleccion_id'])\
                           .agg({'votos_cantidad': 'sum'}).reset_index()



In [ ]:

# More transformations...
# data_circ = data_aggregated.groupby(['eleccion_id', 'cargo_id', 'agrupacion_nombre_', 'lista_numero', 'votos_tipo', 'distrito_id', 'seccion_id', 'circuito_id'])[['votos_cantidad']].sum()
data_circ = data_aggregated.groupby(['eleccion_id', 'cargo_id', 'agrupacion_nombre_', 'votos_tipo', 'distrito_id', 'seccion_id', 'circuito_id'])[['votos_cantidad']].sum()
data_circ = data_circ.reset_index()
data_circ = data_circ.merge(eleccion_tags).merge(cargo)



In [ ]:

# Group by 'eleccion_tag', 'cargo_tag', and 'in1_& (df.distrito_id == 2)prov', and calculate the sum of 'votos_cantidad', divide for PCT
sum_votes = data_circ.groupby(['eleccion_tag', 'cargo_tag', 'distrito_id', 'seccion_id', 'votos_tipo', 'circuito_id'])['votos_cantidad'].transform('sum')
data_circ['votos_porcentaje'] = data_circ['votos_cantidad'] / sum_votes
votos_agrup_circ = data_circ.reset_index(drop = True)

# votos_agrup_lista_circ.to_csv('./../datos/out/votos_agrup_lista_circ.csv', index = False)

# votos_agrup_lista_circ = pd.read_csv('./../datos/out/votos_agrup_lista_circ.csv')



### 3. Análisis Exploratorio de Datos (EDA)

#### Resumen Estadístico:
- No se proporciona un resumen estadístico explícito de los datos en el código proporcionado.

#### Visualización de Datos:
- No se incluyen visualizaciones de datos en el código proporcionado.

#### Identificación de Tendencias:
- Se identifican las agrupaciones y listas principales en términos de votos.



In [ ]:
path_datos = '/content/drive/My Drive/electoral/datos'

In [ ]:

# --- Load Data ---
print("Loading data...")

# Load necessary datasets
radio_region = pd.read_csv(path_datos + '/info/radio_ref.csv', usecols=['radio', 'NOMDPTO', 'Region'])
radios_circuitos_secciones = pd.read_csv(path_datos + '/info/radios_circuitos_secciones_ref.csv')[['COD_2010', 'distrito_id', 'seccion_id', 'seccion_nombre']]
prov_nams = pd.read_csv(path_datos + '/BD/distrito_table.csv')

# --- Data Preprocessing ---
print("Processing data...")

# Radio region processing
radio_region['COD_2010'] = radio_region['radio'].astype(str).str.zfill(9)

# Merge data
merge_data = radios_circuitos_secciones.merge(radio_region, on='COD_2010', how='left')
seccion_region = merge_data.drop(['COD_2010', 'radio'], axis=1).drop_duplicates()
seccion_region = seccion_region.groupby(['distrito_id', 'seccion_id', 'seccion_nombre']).first().reset_index()

# Votos agrup processing
votos = votos_agrup_circ
# data_circ_ix = votos.set_index(['distrito_id', 'seccion_id', 'circuito_id', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_', 'lista_numero', 'votos_tipo'])
data_circ_ix = votos.set_index(['distrito_id', 'seccion_id', 'circuito_id', 'eleccion_tag', 'cargo_tag', 'agrupacion_nombre_', 'votos_tipo'])
# votos_circuito = data_circ_ix['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_', 'lista_numero'])['PASO23n']['PR'].sum(1).sort_values(ascending=False)
votos_circuito = data_circ_ix['votos_cantidad'].unstack(['eleccion_tag', 'cargo_tag', 'votos_tipo', 'agrupacion_nombre_'])['GRAL23n']['PR'].sum(1).sort_values(ascending=False)
circuitos_ppales = votos_circuito[votos_circuito > 1000].index.to_frame().reset_index(drop=True)

# Lista processing
# votos_lista = votos.groupby(['eleccion_tag', 'votos_tipo', 'agrupacion_nombre_', 'lista_numero'])['votos_cantidad'].sum().sort_values(ascending=False)
votos_lista = votos.groupby(['eleccion_tag', 'votos_tipo', 'agrupacion_nombre_'])['votos_cantidad'].sum().sort_values(ascending=False)

N = 20
main_listas = votos_lista.head(N).index.to_frame().reset_index(drop=True)
display(main_listas)

# --- Merge Information ---
print("Merging data...")

info = circuitos_ppales.merge(votos)
info = main_listas.merge(info)
info = info.merge(prov_nams).merge(seccion_region, how='left')

info.head()


Loading data...
Processing data...


,eleccion_tag,votos_tipo,agrupacion_nombre_
0,GRAL23n,POSITIVO,Union Por La Patria
1,GRAL23n,POSITIVO,La Libertad Avanza
2,GRAL23n,POSITIVO,Juntos Por El Cambio
3,PASO23n,POSITIVO,Juntos Por El Cambio
4,PASO23n,POSITIVO,Union Por La Patria
5,PASO23n,POSITIVO,La Libertad Avanza
6,GRAL23n,BLANCO,Resto
7,PASO23n,BLANCO,Resto
8,GRAL23n,POSITIVO,Hacemos Por Nuestro Pais
9,GRAL23n,POSITIVO,Frente De Izquierda Y De Trabajadores - Unidad


Merging data...


,eleccion_tag,votos_tipo,agrupacion_nombre_,distrito_id,seccion_id,circuito_id,eleccion_id,cargo_id,votos_cantidad,año,eleccion_tipo,recuento_tipo,padron_tipo,cargo_nombre,cargo_tag,votos_porcentaje,distrito_nombre,seccion_nombre,NOMDPTO,Region
0,GRAL23n,POSITIVO,Union Por La Patria,2,77,00652A,18,1,90908,2023,GENERAL,PROVISORIO,NORMAL,Presidente,PR,0.487570,Buenos Aires,Merlo,Merlo,Gran Buenos Aires
1,GRAL23n,POSITIVO,Union Por La Patria,2,77,00652A,18,3,85939,2023,GENERAL,PROVISORIO,NORMAL,Diputado Nacional,DN,0.491442,Buenos Aires,Merlo,Merlo,Gran Buenos Aires
2,GRAL23n,POSITIVO,Union Por La Patria,2,77,00652A,18,4,85604,2023,GENERAL,PROVISORIO,NORMAL,Gobernador,GB,0.494998,Buenos Aires,Merlo,Merlo,Gran Buenos Aires
3,GRAL23n,POSITIVO,Union Por La Patria,2,61,00635B,18,1,83053,2023,GENERAL,PROVISORIO,NORMAL,Presidente,PR,0.544374,Buenos Aires,La Matanza,La Matanza,Gran Buenos Aires
4,GRAL23n,POSITIVO,Union Por La Patria,2,61,00635B,18,3,79912,2023,GENERAL,PROVISORIO,NORMAL,Diputado Nacional,DN,0.549771,Buenos Aires,La Matanza,La Matanza,Gran Buenos Aires



### 4. Modelado e Inferencia

**Implementacion Inferencia Ecologica King**


In [ ]:
path_datos = '/content/drive/My Drive/electoral/datos'

agg_circuitos_CONDACT = pd.read_csv(path_datos + '/EPH/agg_circuitos_CONDACT.csv', index_col=0, dtype={'circuito': str, 'PROV': int})
agg_circuitos_P02 = pd.read_csv(path_datos + '/EPH/agg_circuitos_P02.csv', index_col=0, dtype={'circuito': str, 'PROV': int})

In [ ]:
import pandas as pd

demographics = agg_circuitos_P02
demographics = demographics.set_index(['PROV_REF_ID', 'circuito', 'P02']).unstack()['P02_count']
demographics = demographics.div(demographics.sum(1), 0)

voting_outcomes = info[['distrito_id', 'seccion_id', 'circuito_id', 'eleccion_tag', 'votos_tipo', 'agrupacion_nombre_', 'lista_numero', 'cargo_tag', 'votos_cantidad', 'votos_porcentaje']]
voting_outcomes = voting_outcomes.set_index(['distrito_id', 'circuito_id', 'eleccion_tag', 'votos_tipo', 'agrupacion_nombre_', 'lista_numero', 'cargo_tag']).sort_index()['votos_porcentaje'].unstack(['eleccion_tag', 'votos_tipo', 'agrupacion_nombre_', 'lista_numero', 'cargo_tag'])['PASO23n']



ValueError: ignored

In [ ]:
demographics['prop_men'] = demographics['men'] / (demographics['men'] + demographics['women'])
voting_outcomes['prop_votes_for_party'] = voting_outcomes['votes_for_party'] / voting_outcomes['total_votes']


KeyError: ignored

In [ ]:
data = pd.merge(demographics, voting_outcomes, on=['distrito_id', 'seccion_id', 'circuito_id'])

NameError: ignored


#### Selección de Modelo:
- Se implementa un modelo de inferencia ecológica, aunque no se especifica el modelo exacto o la justificación de su elección.

### Likelihood

Assuming the TOM distribution's likelihood function has a specific form, say $f(x;θ,τ)$, the log likelihood for the entire dataset is the sum of the likelihoods for each data point:

$$\log L(\theta, \tau) = \sum_{i=1}^{n} \log f(x_i; \theta, \tau)$$


TOM distribution: $$\exp(-(x-\theta)^2 / (2 \tau^2))$$

##### **Modulos**

In [ ]:
from scipy.stats import betabinom

def beta_binomial_likelihood(params, k, n):
    theta, tau = params
    likelihood_men = betabinom.pmf(k, n, theta * tau, (1 - theta) * tau)
    likelihood_women = betabinom.pmf(k, n, theta * (1 - tau), (1 - theta) * (1 - tau))
    return likelihood_men * likelihood_women


# %%
import numpy as np

def objective(params, k, n):
    return -np.sum(np.log(beta_binomial_likelihood(params, k, n)))


# %%
from scipy.optimize import minimize

#### Ajuste de Modelo:
- Se realizan cálculos y transformaciones en los datos demográficos y de votación para prepararlos para el análisis de inferencia ecológica.


In [ ]:

initial_guess = [0.5, 0.5]
bounds = [(0, 1), (0, 1)]
result = minimize(objective, initial_guess, args=(data['votes_for_party'], data['total_votes']), bounds=bounds)
theta_est, tau_est = result.x


# %%
print(f"It is estimated that {theta_est * 100:.2f}% of men vote for the party.")
print(f"It is estimated that {(1 - theta_est) * 100:.2f}% of women vote for the party.")


In [ ]:

# %%
from scipy.optimize import minimize
import numpy as np

def tom_distribution(x, theta, tau):
    """
    This is a placeholder for the TOM distribution.
    It should return the likelihood of x given parameters theta and tau.
    """
    # Hypothetical form of the TOM distribution
    return np.exp(-(x-theta)**2 / (2*tau**2))

def tom_likelihood(params, data):
    # Extract parameters
    theta, tau = params

    # Compute likelihood based on TOM distribution
    likelihoods = tom_distribution(data, theta, tau)

    # Compute the log-likelihood
    log_likelihood = np.sum(np.log(likelihoods))

    return -log_likelihood  # We want to maximize likelihood, so return the negative


# %%

values_to_model = data['prop_votes_for_party'].tolist()
initial_guess = [0, 1]  # Initial guess for theta and tau
result = minimize(tom_likelihood, initial_guess, args=(values_to_model,))
theta_est, tau_est = result.x

print("Estimated theta:", theta_est)
print("Estimated tau:", tau_est)




#### Evaluación de Modelo:
- No se proporciona una evaluación explícita del modelo en el código proporcionado.


#### Inferencias:
- Se realizan inferencias sobre los resultados de votación, aunque no se detallan explícitamente las conclusiones o interpretaciones.

### 5. Interpretación de Resultados


### 6. Visualización de Resultados
